# PHPP Data Preprocessing

### Real-World Fisheries Operational Data — PT. Daya Bahari Nusantara

This notebook documents the preprocessing workflow for **Post-Production Fisheries Levy (Pungutan Hasil Perikanan Pascaproduksi / PHPP)** data used in the practical-work project at **PT. Daya Bahari Nusantara**.

The original records were manually compiled from operational **Clearance In** and **Clearance Out** documents and later used for exploratory data analysis, statistical testing, regression analysis, clustering, and data visualization.

### Preprocessing objectives

- Load and inspect the raw operational dataset.
- Validate the dataset structure and data quality.
- Standardize departure and arrival date fields.
- Encode categorical variables while preserving their mappings.
- Export a consistent preprocessed dataset for downstream analysis.

> **Portfolio / data-governance note:** This project is based on real operational records. If explicit permission to publish identifiable records has not been granted, company, vessel, and other entity identifiers should be anonymized before the dataset or executed notebook outputs are made public.


## 1. Environment Setup and Data Loading

The raw PHPP dataset is stored in an Excel workbook. This step imports `pandas`, loads the dataset, and previews the first observations to verify that the file has been read correctly.

The source dataset contains operational attributes related to companies, vessels, fishing activities, catch volume, production value, and fisheries levies.


In [ ]:
import pandas as pd

DATA_PATH = "data/DataPHPP.xlsx"

data = pd.read_excel(DATA_PATH)

print(f"Dataset shape: {data.shape[0]:,} rows × {data.shape[1]} columns")
data.head()


## 2. Initial Data Understanding

Before applying transformations, the dataset structure is reviewed to understand:

- column names;
- data types;
- number of non-null observations;
- memory usage; and
- whether the imported schema is consistent with the expected PHPP records.

The original dataset used in this project contains **777 observations and 12 variables**.


In [ ]:
data.info()


### 2.1 Basic Data-Quality Check

A compact data-quality summary is created to inspect the number of missing values and unique values in each column. Full-row duplicates are also checked before any transformation is applied.

This step is intentionally diagnostic: no rows are removed automatically because repeated operational attributes can be legitimate when one vessel trip records multiple fish species.


In [ ]:
quality_summary = pd.DataFrame({
    "dtype": data.dtypes.astype(str),
    "missing_values": data.isna().sum(),
    "unique_values": data.nunique(dropna=False)
})

display(quality_summary)

print(f"Full-row duplicates: {data.duplicated().sum()}")


## 3. Standardize Date Columns

`Tanggal Keberangkatan` and `Tanggal Kedatangan` were recorded using mixed date representations in the source documents.

Both columns are converted to `datetime` using `pandas.to_datetime()` so they can be used reliably in downstream time-based analysis, such as:

- trip-duration calculations;
- monthly and yearly activity patterns;
- departure and arrival trends; and
- operational seasonality.

The columns are intentionally retained as `datetime` objects rather than converted back to formatted strings.


In [ ]:
date_columns = ["Tanggal Keberangkatan", "Tanggal Kedatangan"]

for column in date_columns:
    data[column] = pd.to_datetime(
        data[column],
        errors="coerce",
        dayfirst=True
    )

data[date_columns].head()


### 3.1 Validate Date Conversion

After conversion, the number of invalid or unparsed dates is checked. A successful transformation should not introduce unexpected missing dates.


In [ ]:
date_validation = pd.DataFrame({
    "column": date_columns,
    "invalid_or_missing_dates": [data[col].isna().sum() for col in date_columns]
})

date_validation


## 4. Validate Data Types After Transformation

The dataset schema is inspected again to confirm that the departure and arrival columns have changed from generic object fields to `datetime64[ns]`.

This validation step makes the preprocessing workflow easier to audit and reproduce.


In [ ]:
data.info()


## 5. Encode Categorical Variables

Four categorical variables are encoded using `LabelEncoder`:

- `Nama Perusahaan`
- `Nama Kapal`
- `Alat Penangkapan Ikan`
- `Jenis Ikan`

A separate copy named `data_encoded` is created so the original human-readable dataframe remains available during analysis.

Each encoder is also stored in `label_encoders`, allowing the numerical codes to be mapped back to their original categories.

> **Important:** Label-encoded values are category identifiers, not ordinal measurements. A larger code does not imply that one company, vessel, fishing gear, or fish species is quantitatively “greater” than another.


In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_columns = [
    "Nama Perusahaan",
    "Nama Kapal",
    "Alat Penangkapan Ikan",
    "Jenis Ikan"
]

data_encoded = data.copy()
label_encoders = {}

for column in categorical_columns:
    encoder = LabelEncoder()
    data_encoded[column] = encoder.fit_transform(data_encoded[column])
    label_encoders[column] = encoder

data_encoded.head()


## 6. Review Encoding Mappings

The number of categories and the mapping generated for each encoded feature are reviewed to maintain interpretability and reproducibility.

The mappings can later be used to translate analytical results back into meaningful company, vessel, fishing-gear, or fish-species labels.


In [ ]:
for column, encoder in label_encoders.items():
    mapping = dict(zip(encoder.classes_, range(len(encoder.classes_))))

    print(f"{column}: {len(mapping)} categories")
    print(mapping)
    print("-" * 80)


## 7. Export the Preprocessed Dataset

The transformed dataset is exported to CSV for use in the main analytical notebook.

The export preserves:

- standardized datetime columns;
- numerical categorical identifiers; and
- all original numerical PHPP variables.

No observations are intentionally removed in this preprocessing notebook.


In [ ]:
OUTPUT_PATH = "data/dataphpp2.csv"

data_encoded.to_csv(OUTPUT_PATH, index=False)

print(f"Preprocessed dataset saved to: {OUTPUT_PATH}")
print(f"Final shape: {data_encoded.shape[0]:,} rows × {data_encoded.shape[1]} columns")


## 8. Preprocessing Summary

The preprocessing stage prepares the manually collected PHPP operational records for reproducible downstream analysis.

**Completed steps:**

1. Loaded the raw Excel dataset.
2. Reviewed the dataset schema and basic data quality.
3. Standardized departure and arrival dates into `datetime`.
4. Validated the transformed date fields.
5. Encoded four categorical variables while retaining reversible mappings.
6. Exported the processed dataset for the main EDA workflow.

Outlier assessment and analytical transformations are intentionally handled in the main analysis notebook, where decisions can be made in the context of each statistical test, visualization, regression, or clustering task.
